# JWST / MAST Deep Analytics Notebook

This notebook is a **detailed, end-to-end analytics workflow** for public **JWST** data from **MAST**.

It is designed to answer:

- How do we **grab public JWST data from MAST**?
- What **scientific and operational information and analysis** can we extract from the captured data?
- How do we move from **observation search** to **download** to **analysis** to a **report**?

---

## What this notebook covers

### 1. Data acquisition from MAST
- connectivity checks
- target-based search
- observation-level filtering
- product-level filtering
- public-only download workflow

### 2. Observation and product analytics
- observation inventory
- instrument and detector distribution
- filter distribution
- exposure-time summaries
- calibration/data-rights/product-type summaries

### 3. FITS image analytics
- HDU discovery
- science image selection
- background estimation
- hot-pixel mitigation
- global image statistics
- brightest-source localization
- centroiding
- aperture photometry
- radial profiles
- row/column detector diagnostics
- percentile contrast diagnostics

### 4. Multi-filter science analysis
- per-filter comparisons
- WCS alignment
- cross-filter flux comparison
- simple color-style metrics
- optional RGB quicklook

### 5. Report exports
- CSV summaries
- JSON summary report
- figure exports
- machine-readable inventory tables

---

## What information can we take from JWST / MAST data?

From the downloaded observations and FITS products, we can often extract:

### Observation-level information
- target name
- RA / Dec
- observing program ID
- instrument
- detector
- proposal metadata
- observation date/time proxies
- exposure time
- data rights
- product type
- calibration level

### Image-level information
- image dimensions
- pixel intensity statistics
- sky/background level
- noise estimates
- bright-source positions
- approximate source fluxes
- radial light concentration
- filter-dependent brightness differences
- detector striping / row-column structure
- WCS footprint and alignment quality

### Cross-filter information
- how morphology changes with wavelength
- which structures dominate in short vs long wavelength filters
- which band is brightest for the chosen target
- simple color trends across filters
- approximate flux ranking across bands

---

## Recommended public targets
Good starter targets for public JWST imaging include:
- `SMACS 0723`
- `NGC 346`
- `M16`
- `Carina`
- `NGC 628`
- `NGC 3324`

---

## Notes
- This notebook targets **public data only**.
- It tries to prefer **image-like calibrated products**.
- It is intentionally verbose and analytics-oriented.
- It is a framework for exploration, not a final publication pipeline.

In [ ]:
# Run this once in a clean environment if needed.
%pip -q install -U astroquery astropy photutils reproject scikit-image matplotlib scipy pandas requests pillow

## Imports

In [ ]:
from pathlib import Path
import json
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from astropy.io import fits
from astropy.table import Table, vstack, unique
from astropy.stats import sigma_clipped_stats
from astropy.visualization import simple_norm, make_lupton_rgb
from astropy.wcs import WCS

from astroquery.mast import Observations
from scipy.ndimage import median_filter, center_of_mass
from skimage.feature import peak_local_max

from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from photutils.segmentation import detect_threshold, detect_sources, SourceCatalog

from reproject import reproject_interp

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["image.origin"] = "lower"

print("Imported successfully.")

## Configuration

In [ ]:
# =========================
# Core search configuration
# =========================
MAST_TARGET = "SMACS 0723"     # Change this target to any public JWST target you want
MAST_RADIUS_DEG = 0.03
MAST_MAX_OBS = 5
PRODUCT_LIMIT = 8

# =========================
# Filtering options
# =========================
JWST_ONLY = True
PUBLIC_ONLY = True
IMAGE_ONLY = True

PREFERRED_SUBGROUPS = ["I2D", "DRZ", "DRC", "CAL", "SCI"]
PREFERRED_FILENAME_KEYWORDS = ["i2d", "drz", "drc", "cal", "rate", "sci"]

# =========================
# Output folders
# =========================
BASE_DIR = Path(".")
DOWNLOAD_DIR = BASE_DIR / "mast_downloads"
FIG_DIR = BASE_DIR / "figures"
OUTPUT_DIR = BASE_DIR / "output"

for folder in [DOWNLOAD_DIR, FIG_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# =========================
# Analysis parameters
# =========================
BACKGROUND_SIGMA = 3.0
BACKGROUND_MAXITERS = 10
LOW_PERCENTILE = 1.0
HIGH_PERCENTILE = 99.7
STRETCH = "asinh"

# =========================
# Photometry parameters
# =========================
APER_R = 6.0
ANN_R_IN = 8.0
ANN_R_OUT = 12.0

# =========================
# Source detection parameters
# =========================
SEG_NSIGMA = 2.5
SEG_NPIXELS = 15
MAX_PEAKS_PER_IMAGE = 50

# =========================
# RGB options
# =========================
RGB_ENABLED = True
USE_LUPTON_RGB = False

# =========================
# Download options
# =========================
ENABLE_CLOUD_DOWNLOAD = True
CACHE_DOWNLOADS = True
HTTP_TIMEOUT = 45

print("Target:", MAST_TARGET)

## Connectivity checks

In [ ]:
def check_url(url, timeout=20):
    try:
        r = requests.get(url, timeout=timeout)
        return {"url": url, "ok": True, "status_code": r.status_code, "detail": "reachable"}
    except Exception as e:
        return {"url": url, "ok": False, "status_code": None, "detail": str(e)}

checks = [
    check_url("https://mast.stsci.edu"),
    check_url("https://archive.stsci.edu"),
]

connectivity_df = pd.DataFrame(checks)
display(connectivity_df)

if not connectivity_df["ok"].all():
    raise RuntimeError(
        "Could not reach one or more STScI / MAST endpoints. "
        "This usually means a notebook runtime internet problem rather than an astronomy-code problem."
    )

## Optional cloud-backed public download mode

In [ ]:
if ENABLE_CLOUD_DOWNLOAD:
    try:
        Observations.enable_cloud_dataset()
        print("Cloud-backed public downloads enabled.")
    except Exception as e:
        print("Could not enable cloud-backed downloads:", e)
        print("Continuing with normal MAST downloads.")

## Utility functions

In [ ]:
def choose_science_hdu(hdul):
    candidates = []
    for idx, hdu in enumerate(hdul):
        data = getattr(hdu, "data", None)
        if data is None or not isinstance(data, np.ndarray) or data.ndim < 2:
            continue
        hdr = hdu.header
        name = (getattr(hdu, "name", "") or "").upper()
        score = 0
        if name == "SCI":
            score += 100
        if data.ndim == 2:
            score += 25
        if idx == 0:
            score += 10
        if "CTYPE1" in hdr or "WCSAXES" in hdr:
            score += 30
        if data.shape[-2] > 128 and data.shape[-1] > 128:
            score += 20
        candidates.append((score, idx, hdu))
    if not candidates:
        return None, None, None
    candidates.sort(reverse=True, key=lambda x: x[0])
    _, idx, hdu = candidates[0]
    return idx, np.array(hdu.data), hdu.header


def robust_clean(data):
    x = np.array(data, dtype=np.float32)
    bad = ~np.isfinite(x)
    if bad.any():
        good = ~bad
        fill = np.nanmedian(x[good]) if good.any() else 0.0
        x[bad] = fill
    return x


def background_subtract(data, sigma=BACKGROUND_SIGMA, maxiters=BACKGROUND_MAXITERS):
    mean, median, std = sigma_clipped_stats(data, sigma=sigma, maxiters=maxiters)
    return data - median, {"mean": float(mean), "median": float(median), "std": float(std)}


def remove_hot_pixels(data, size=3):
    med = median_filter(data, size=size)
    resid = data - med
    _, _, std = sigma_clipped_stats(resid, sigma=3.0)
    mask = resid > 8 * std
    out = data.copy()
    out[mask] = med[mask]
    return out


def brightest_pixel_xy(data):
    iy, ix = np.unravel_index(np.nanargmax(data), data.shape)
    return int(ix), int(iy)


def centroid_near_peak(data, peak_xy, half_size=12):
    x0, y0 = peak_xy
    y1 = max(0, y0 - half_size)
    y2 = min(data.shape[0], y0 + half_size + 1)
    x1 = max(0, x0 - half_size)
    x2 = min(data.shape[1], x0 + half_size + 1)
    cut = data[y1:y2, x1:x2]
    shifted = cut - np.nanmin(cut)
    if np.allclose(shifted.sum(), 0):
        return float(x0), float(y0)
    cy, cx = center_of_mass(shifted)
    return float(x1 + cx), float(y1 + cy)


def aperture_photometry_with_background(data, x, y, r=APER_R, r_in=ANN_R_IN, r_out=ANN_R_OUT):
    pos = [(x, y)]
    aper = CircularAperture(pos, r=r)
    ann = CircularAnnulus(pos, r_in=r_in, r_out=r_out)

    aper_tbl = aperture_photometry(data, aper)
    ann_tbl = aperture_photometry(data, ann)

    aper_sum = float(aper_tbl["aperture_sum"][0])
    ann_sum = float(ann_tbl["aperture_sum"][0])

    aper_area = float(aper.area)
    ann_area = float(ann.area)
    bkg_mean = ann_sum / ann_area if ann_area > 0 else 0.0
    net_flux = aper_sum - bkg_mean * aper_area

    return {
        "x": float(x),
        "y": float(y),
        "aperture_r": float(r),
        "annulus_r_in": float(r_in),
        "annulus_r_out": float(r_out),
        "aperture_sum": aper_sum,
        "annulus_sum": ann_sum,
        "background_mean_per_pix": bkg_mean,
        "net_flux": net_flux,
    }


def radial_profile(data, center=None, binsize=1):
    y, x = np.indices(data.shape)
    if center is None:
        center = (data.shape[1] / 2, data.shape[0] / 2)
    cx, cy = center
    r = np.sqrt((x - cx)**2 + (y - cy)**2)
    rbin = (r / binsize).astype(int)
    tbin = np.bincount(rbin.ravel(), data.ravel())
    nr = np.bincount(rbin.ravel())
    profile = np.divide(tbin, nr, out=np.zeros_like(tbin, dtype=float), where=nr > 0)
    radius = np.arange(len(profile)) * binsize
    return radius, profile


def detect_filter_name(header, path=None):
    keys = ["FILTER", "PUPIL", "FILTNAM1", "FILTNAM2", "FILTER1", "FILTER2"]
    vals = []
    for k in keys:
        if k in header and str(header[k]).strip():
            vals.append(str(header[k]).strip().lower())
    if path is not None:
        stem = Path(path).stem.lower()
        vals.extend(re.findall(r"f\d{3}[wmn]", stem))
    vals = [v for v in vals if v not in {"clear", "none", "nan"}]
    if vals:
        for v in vals:
            if re.match(r"f\d{3}[wmn]", v):
                return v
        return vals[0]
    return Path(path).stem.lower() if path is not None else "unknown"


def wavelength_key(name):
    m = re.search(r"f(\d{3})([wmn])", str(name).lower())
    return int(m.group(1)) if m else 9999


def auto_assign_rgb(filter_names):
    ordered = sorted(filter_names, key=wavelength_key)
    n = len(ordered)
    if n == 1:
        return {"R": ordered[0], "G": ordered[0], "B": ordered[0]}
    if n == 2:
        return {"R": ordered[1], "G": ordered[0], "B": ordered[0]}
    if n == 3:
        return {"B": ordered[0], "G": ordered[1], "R": ordered[2]}
    return {"B": ordered[0], "G": ordered[n // 2], "R": ordered[-1]}


def normalize_channel(data, low=LOW_PERCENTILE, high=HIGH_PERCENTILE, stretch=STRETCH):
    x = np.array(data, dtype=np.float32)
    lo = np.nanpercentile(x, low)
    hi = np.nanpercentile(x, high)
    if hi <= lo:
        hi = lo + 1e-6
    x = np.clip((x - lo) / (hi - lo), 0, 1)
    if stretch == "sqrt":
        x = np.sqrt(x)
    elif stretch == "log":
        x = np.log1p(1000 * x) / np.log1p(1000)
    elif stretch == "asinh":
        a = 8.0
        x = np.arcsinh(a * x) / np.arcsinh(a)
    elif stretch != "linear":
        raise ValueError(f"Unknown stretch: {stretch}")
    return np.clip(x, 0, 1)


def image_summary_stats(data):
    return {
        "min": float(np.nanmin(data)),
        "max": float(np.nanmax(data)),
        "mean": float(np.nanmean(data)),
        "median": float(np.nanmedian(data)),
        "std": float(np.nanstd(data)),
        "p01": float(np.nanpercentile(data, 1)),
        "p05": float(np.nanpercentile(data, 5)),
        "p50": float(np.nanpercentile(data, 50)),
        "p95": float(np.nanpercentile(data, 95)),
        "p99": float(np.nanpercentile(data, 99)),
    }


def save_current_figure(path):
    plt.savefig(path, bbox_inches="tight", dpi=160)
    print("Saved:", path)

## Step 1 — Search public JWST observations

This gives an **observation inventory**.  
That means we can start learning:

- what observations exist near the target
- which instruments were used
- whether they are public
- whether they are imaging products
- how many candidate observation groups we may analyze

In [ ]:
def search_public_jwst_observations(target=MAST_TARGET, radius_deg=MAST_RADIUS_DEG, max_obs=MAST_MAX_OBS):
    print(f"Searching MAST around target={target!r} with radius={radius_deg} deg ...")
    obs = Observations.query_object(target, radius=f"{radius_deg} deg")
    if len(obs) == 0:
        raise RuntimeError("No observations found for this target/radius.")

    df = obs.to_pandas()

    if JWST_ONLY and "obs_collection" in df.columns:
        df = df[df["obs_collection"].astype(str).str.upper() == "JWST"]

    if PUBLIC_ONLY and "dataRights" in df.columns:
        rights = df["dataRights"].astype(str).str.upper()
        df = df[(rights == "PUBLIC") | (rights == "")]

    if IMAGE_ONLY and "dataproduct_type" in df.columns:
        dtype = df["dataproduct_type"].astype(str).str.lower()
        df = df[dtype.str.contains("image", na=False) | dtype.eq("")]

    if len(df) == 0:
        raise RuntimeError("No observations remained after JWST/public/image filtering.")

    sort_cols = [c for c in ["t_min", "proposal_id", "obs_id"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols, ascending=False)

    key_col = "obsid" if "obsid" in df.columns else df.columns[0]
    df = df.drop_duplicates(subset=[key_col]).head(max_obs)

    selected = obs[np.isin(obs["obsid"], df["obsid"].tolist())]
    print(f"Selected {len(selected)} observation(s).")
    return selected, df

obs_rows, obs_df = search_public_jwst_observations()
display(obs_df.head(20))

## Observation inventory analytics

In [ ]:
obs_inventory_cols = [c for c in [
    "target_name", "obs_id", "obsid", "proposal_id", "instrument_name",
    "filters", "t_exptime", "t_min", "dataRights", "dataproduct_type"
] if c in obs_df.columns]

display(obs_df[obs_inventory_cols].head(20))

print("Observation count:", len(obs_df))

for col in ["instrument_name", "dataRights", "dataproduct_type", "filters"]:
    if col in obs_df.columns:
        print(f"\nValue counts for {col}:")
        display(obs_df[col].astype(str).value_counts().rename_axis(col).reset_index(name="count"))

In [ ]:
# Exposure-time analytics if available
if "t_exptime" in obs_df.columns:
    exptime = pd.to_numeric(obs_df["t_exptime"], errors="coerce").dropna()
    if len(exptime):
        exptime_summary = pd.DataFrame([{
            "count": len(exptime),
            "min_s": exptime.min(),
            "median_s": exptime.median(),
            "mean_s": exptime.mean(),
            "max_s": exptime.max(),
            "std_s": exptime.std(),
        }])
        display(exptime_summary)

        plt.figure(figsize=(8, 4))
        plt.hist(exptime, bins=min(15, max(5, len(exptime))))
        plt.xlabel("Exposure time [s]")
        plt.ylabel("Count")
        plt.title("Exposure time distribution across selected observations")
        plt.grid(True, alpha=0.3)
        plt.show()

## Step 2 — Gather product lists

This moves from the **observation layer** to the **product layer**.

At this stage, we can inspect:
- how many total files are attached to the selected observations
- which file types exist
- which calibration levels dominate
- which subgroups look scientifically useful for imaging

In [ ]:
def gather_products_for_observations(obs_rows):
    tables = []
    for i, row in enumerate(obs_rows):
        print(f"Fetching products for observation {i+1}/{len(obs_rows)} ...")
        prod = Observations.get_product_list(row)
        if len(prod):
            tables.append(prod)
    if not tables:
        raise RuntimeError("No products returned for selected observations.")
    merged = unique(vstack(tables), keys="dataURI" if "dataURI" in tables[0].colnames else "productFilename")
    return merged

products = gather_products_for_observations(obs_rows)
print("Raw product count:", len(products))
products[:5]

## Product-level analytics

In [ ]:
products_df = products.to_pandas()
display(products_df.head(20))

for col in ["productType", "productSubGroupDescription", "extension", "dataRights", "calib_level"]:
    if col in products_df.columns:
        print(f"\nDistribution of {col}:")
        display(products_df[col].astype(str).value_counts().rename_axis(col).reset_index(name="count"))

In [ ]:
if "size" in products_df.columns:
    size_series = pd.to_numeric(products_df["size"], errors="coerce").dropna()
    if len(size_series):
        size_summary = pd.DataFrame([{
            "count": len(size_series),
            "min_bytes": size_series.min(),
            "median_bytes": size_series.median(),
            "mean_bytes": size_series.mean(),
            "max_bytes": size_series.max(),
        }])
        display(size_summary)

## Step 3 — Filter to a useful FITS subset for deep analysis

The goal is not to download everything.  
The goal is to download a **small, stable, analytically useful set**.

In [ ]:
def filter_products_for_download(products, product_limit=PRODUCT_LIMIT):
    filtered = products

    try:
        filtered = Observations.filter_products(
            filtered,
            extension=["fits", "fits.gz"],
            productType=["SCIENCE", "PRODUCT", "!AUXILIARY"]
        )
    except Exception as e:
        print("filter_products warning:", e)

    pdf = filtered.to_pandas()

    if len(pdf) == 0:
        raise RuntimeError("No products left after FITS/science filtering.")

    for col in ["productFilename", "productSubGroupDescription", "dataRights", "calib_level", "productType"]:
        if col not in pdf.columns:
            pdf[col] = ""

    pdf["productFilename_l"] = pdf["productFilename"].astype(str).str.lower()
    pdf["subgroup_l"] = pdf["productSubGroupDescription"].astype(str).str.upper()
    pdf["rights_u"] = pdf["dataRights"].astype(str).str.upper()
    pdf["ptype_u"] = pdf["productType"].astype(str).str.upper()

    fits_mask = pdf["productFilename_l"].str.endswith(".fits") | pdf["productFilename_l"].str.endswith(".fits.gz")
    public_mask = (pdf["rights_u"] == "PUBLIC") | (pdf["rights_u"] == "")
    subgroup_score = pdf["subgroup_l"].isin(PREFERRED_SUBGROUPS).astype(int)
    keyword_score = pdf["productFilename_l"].apply(lambda s: int(any(k in s for k in PREFERRED_FILENAME_KEYWORDS)))
    type_score = pdf["ptype_u"].isin(["SCIENCE", "PRODUCT"]).astype(int)

    pdf = pdf[fits_mask & public_mask].copy()
    if len(pdf) == 0:
        raise RuntimeError("No public FITS files remained after filtering.")

    pdf["rank_score"] = 10 * subgroup_score.loc[pdf.index] + 5 * keyword_score.loc[pdf.index] + 2 * type_score.loc[pdf.index]

    if "size" in pdf.columns:
        pdf["size_num"] = pd.to_numeric(pdf["size"], errors="coerce")
    else:
        pdf["size_num"] = np.nan

    pdf = pdf.sort_values(["rank_score", "size_num"], ascending=[False, True])
    pdf = pdf.drop_duplicates(subset=["productFilename"]).head(product_limit).reset_index(drop=True)

    keep_names = set(pdf["productFilename"].tolist())
    mask = np.array([str(row["productFilename"]) in keep_names for row in filtered], dtype=bool)
    final = filtered[mask]
    return final, pdf

download_products_table, download_products_df = filter_products_for_download(products)
print("Products selected for download:", len(download_products_table))
display(download_products_df)

## Step 4 — Download selected products

In [ ]:
def download_selected_products(products_table, download_dir=DOWNLOAD_DIR, cache=CACHE_DOWNLOADS):
    download_dir.mkdir(parents=True, exist_ok=True)
    manifest_rows = []

    for row in products_table:
        fname = str(row["productFilename"]) if "productFilename" in row.colnames else "unknown.fits"
        uri = str(row["dataURI"]) if "dataURI" in row.colnames else None
        target_path = download_dir / fname

        if uri is None:
            manifest_rows.append({
                "productFilename": fname,
                "status": "ERROR",
                "message": "Missing dataURI",
                "url": None,
                "local_path": None
            })
            continue

        try:
            status, msg, url = Observations.download_file(
                uri,
                local_path=str(target_path),
                cache=cache,
                verbose=True
            )
            manifest_rows.append({
                "productFilename": fname,
                "status": status,
                "message": msg,
                "url": url,
                "local_path": str(target_path) if target_path.exists() else None
            })
        except Exception as e:
            manifest_rows.append({
                "productFilename": fname,
                "status": "ERROR",
                "message": str(e),
                "url": None,
                "local_path": None
            })

    return pd.DataFrame(manifest_rows)

download_manifest = download_selected_products(download_products_table)
display(download_manifest)

ok_files = [Path(p) for p in download_manifest["local_path"].dropna().tolist() if Path(p).exists()]
print("Downloaded FITS files:", len(ok_files))

if len(ok_files) == 0:
    raise RuntimeError("No FITS files downloaded successfully.")

## Step 5 — FITS inventory and metadata extraction

Here we learn:
- how many HDUs each file has
- which HDU looks like the science image
- instrument / detector / telescope metadata
- target labels
- image dimensions
- candidate filter name

In [ ]:
inventory_rows = []

for path in ok_files:
    try:
        with fits.open(path) as hdul:
            sci_idx, data, header = choose_science_hdu(hdul)
            inventory_rows.append({
                "path": str(path),
                "filename": path.name,
                "n_hdus": len(hdul),
                "science_hdu_index": sci_idx,
                "has_usable_2d_science": data is not None,
                "shape": tuple(data.shape) if data is not None else None,
                "filter_name": detect_filter_name(header, path) if data is not None else None,
                "instrument": str(header.get("INSTRUME", "")) if data is not None else "",
                "detector": str(header.get("DETECTOR", "")) if data is not None else "",
                "telescope": str(header.get("TELESCOP", "")) if data is not None else "",
                "target": str(header.get("TARGNAME", "")) if data is not None else "",
                "bunit": str(header.get("BUNIT", "")) if data is not None else "",
            })
    except Exception as e:
        inventory_rows.append({
            "path": str(path),
            "filename": path.name,
            "n_hdus": None,
            "science_hdu_index": None,
            "has_usable_2d_science": False,
            "shape": None,
            "filter_name": None,
            "instrument": "",
            "detector": "",
            "telescope": "",
            "target": "",
            "bunit": "",
            "error": str(e)
        })

catalog = pd.DataFrame(inventory_rows)
display(catalog)

usable_catalog = catalog[catalog["has_usable_2d_science"] == True].copy()
if len(usable_catalog) == 0:
    raise RuntimeError("Downloaded files exist, but no usable 2D science image was found.")

In [ ]:
print("Filter distribution:")
display(usable_catalog["filter_name"].astype(str).value_counts().rename_axis("filter_name").reset_index(name="count"))

print("Instrument distribution:")
display(usable_catalog["instrument"].astype(str).value_counts().rename_axis("instrument").reset_index(name="count"))

print("Detector distribution:")
display(usable_catalog["detector"].astype(str).value_counts().rename_axis("detector").reset_index(name="count"))

## Step 6 — Pick one representative image per filter

This prevents duplicate-heavy analysis and gives a clean
**one-best-image-per-filter** view.

In [ ]:
usable_catalog["n_pix"] = usable_catalog["shape"].apply(lambda s: int(np.prod(s[-2:])))
best_per_filter = (
    usable_catalog.sort_values("n_pix", ascending=False)
    .drop_duplicates("filter_name")
    .reset_index(drop=True)
)

display(best_per_filter[["filter_name", "filename", "science_hdu_index", "shape", "instrument", "detector", "n_pix"]])

## Step 7 — Load cleaned images and compute deep image metrics

For each representative filter image we compute:
- robust background statistics
- cleaned image version
- global summary statistics
- brightest-pixel position
- local centroid
- aperture photometry
- top-peak counts
- simple segmentation catalog size

In [ ]:
loaded = {}
metrics_rows = []
source_rows = []

for _, row in best_per_filter.iterrows():
    filt = row["filter_name"]
    path = row["path"]
    hdu_index = int(row["science_hdu_index"])

    with fits.open(path) as hdul:
        raw = robust_clean(hdul[hdu_index].data)
        header = hdul[hdu_index].header

    bg_sub, bg_stats = background_subtract(raw)
    clean = remove_hot_pixels(bg_sub)

    peak_xy = brightest_pixel_xy(clean)
    cx, cy = centroid_near_peak(clean, peak_xy, half_size=12)
    phot = aperture_photometry_with_background(clean, cx, cy)

    summary = image_summary_stats(clean)

    # quick peak counting
    try:
        peaks = peak_local_max(clean, min_distance=6, num_peaks=MAX_PEAKS_PER_IMAGE)
        n_peaks = int(len(peaks))
    except Exception:
        peaks = np.empty((0, 2), dtype=int)
        n_peaks = 0

    # segmentation-based source catalog
    try:
        threshold = detect_threshold(clean, nsigma=SEG_NSIGMA)
        segm = detect_sources(clean, threshold, npixels=SEG_NPIXELS)
        if segm is not None:
            scatalog = SourceCatalog(clean, segm)
            n_segment_sources = len(scatalog)
        else:
            scatalog = None
            n_segment_sources = 0
    except Exception:
        segm = None
        scatalog = None
        n_segment_sources = 0

    loaded[filt] = {
        "raw": raw,
        "data": clean,
        "header": header,
        "wcs": WCS(header),
        "bg_stats": bg_stats,
        "summary_stats": summary,
        "path": path,
        "peak_xy": peak_xy,
        "centroid_xy": (cx, cy),
        "photometry": phot,
        "peaks": peaks,
        "segmentation": segm,
        "source_catalog": scatalog,
    }

    metrics_rows.append({
        "filter_name": filt,
        "path": path,
        "shape_y": clean.shape[0],
        "shape_x": clean.shape[1],
        "mean": bg_stats["mean"],
        "median": bg_stats["median"],
        "std": bg_stats["std"],
        "clean_min": summary["min"],
        "clean_max": summary["max"],
        "clean_p01": summary["p01"],
        "clean_p50": summary["p50"],
        "clean_p99": summary["p99"],
        "peak_x": peak_xy[0],
        "peak_y": peak_xy[1],
        "centroid_x": cx,
        "centroid_y": cy,
        "net_flux": phot["net_flux"],
        "background_mean_per_pix": phot["background_mean_per_pix"],
        "n_local_peaks": n_peaks,
        "n_segment_sources": n_segment_sources,
    })

    if scatalog is not None and len(scatalog) > 0:
        for j, src in enumerate(scatalog[: min(25, len(scatalog))]):
            source_rows.append({
                "filter_name": filt,
                "source_rank": j + 1,
                "xcentroid": float(src.xcentroid.value if hasattr(src.xcentroid, "value") else src.xcentroid),
                "ycentroid": float(src.ycentroid.value if hasattr(src.ycentroid, "value") else src.ycentroid),
                "area": float(src.area.value if hasattr(src.area, "value") else src.area),
                "segment_flux": float(src.segment_flux.value if hasattr(src.segment_flux, "value") else src.segment_flux),
                "max_value": float(src.max_value.value if hasattr(src.max_value, "value") else src.max_value),
            })

metrics_df = pd.DataFrame(metrics_rows).sort_values("filter_name", key=lambda s: s.map(wavelength_key))
segment_sources_df = pd.DataFrame(source_rows)

display(metrics_df)

## Per-filter preview images

In [ ]:
n = len(loaded)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
if n == 1:
    axes = [axes]

for ax, filt in zip(axes, sorted(loaded.keys(), key=wavelength_key)):
    data = loaded[filt]["data"]
    stats = loaded[filt]["bg_stats"]
    norm = simple_norm(data, stretch=STRETCH, percent=99.5)
    ax.imshow(data, cmap="gray", norm=norm)
    ax.set_title(f"{filt}\nmed={stats['median']:.3g}, std={stats['std']:.3g}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## Source localization and aperture diagnostics

In [ ]:
for filt in sorted(loaded.keys(), key=wavelength_key):
    item = loaded[filt]
    data = item["data"]
    x_peak, y_peak = item["peak_xy"]
    cx, cy = item["centroid_xy"]

    plt.figure(figsize=(7, 7))
    norm = simple_norm(data, stretch=STRETCH, percent=99.5)
    plt.imshow(data, cmap="gray", norm=norm)
    plt.scatter([x_peak], [y_peak], s=90, marker="x", label="brightest pixel")
    plt.scatter([cx], [cy], s=90, marker="+", label="centroid")

    circ = plt.Circle((cx, cy), APER_R, fill=False)
    ann1 = plt.Circle((cx, cy), ANN_R_IN, fill=False, linestyle="--")
    ann2 = plt.Circle((cx, cy), ANN_R_OUT, fill=False, linestyle="--")

    plt.gca().add_patch(circ)
    plt.gca().add_patch(ann1)
    plt.gca().add_patch(ann2)

    plt.legend()
    plt.title(f"Source diagnostics — {filt}")
    plt.axis("off")
    plt.show()

## Radial profile diagnostics

In [ ]:
for filt in sorted(loaded.keys(), key=wavelength_key):
    item = loaded[filt]
    data = item["data"]
    cx, cy = item["centroid_xy"]
    radius, profile = radial_profile(data, center=(cx, cy), binsize=1)

    plt.figure(figsize=(8, 4))
    plt.plot(radius[:250], profile[:250])
    plt.xlabel("Radius [pixels]")
    plt.ylabel("Mean intensity")
    plt.title(f"Radial profile — {filt}")
    plt.grid(True, alpha=0.3)
    plt.show()

## Detector row/column diagnostics

In [ ]:
for filt in sorted(loaded.keys(), key=wavelength_key):
    data = loaded[filt]["data"]
    row_med = np.nanmedian(data, axis=1)
    col_med = np.nanmedian(data, axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(row_med)
    axes[0].set_title(f"Row median profile — {filt}")
    axes[0].set_xlabel("Row index")
    axes[0].set_ylabel("Median signal")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(col_med)
    axes[1].set_title(f"Column median profile — {filt}")
    axes[1].set_xlabel("Column index")
    axes[1].set_ylabel("Median signal")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Segmentation and detected-source analytics

In [ ]:
if len(segment_sources_df):
    print("Detected source catalog preview:")
    display(segment_sources_df.head(25))

    source_count_df = (
        segment_sources_df.groupby("filter_name")
        .size()
        .reset_index(name="listed_segment_sources")
        .sort_values("filter_name", key=lambda s: s.map(wavelength_key))
    )
    display(source_count_df)
else:
    print("No segmentation source rows were produced.")

In [ ]:
# Show segmentation overlays when available
for filt in sorted(loaded.keys(), key=wavelength_key):
    item = loaded[filt]
    data = item["data"]
    segm = item["segmentation"]

    if segm is None:
        print(f"No segmentation map for {filt}")
        continue

    plt.figure(figsize=(7, 7))
    norm = simple_norm(data, stretch=STRETCH, percent=99.5)
    plt.imshow(data, cmap="gray", norm=norm)
    plt.imshow(segm.data, alpha=0.25)
    plt.title(f"Segmentation overlay — {filt}")
    plt.axis("off")
    plt.show()

## Step 8 — Cross-filter comparison table

This table helps answer:
- Which filter has the highest approximate flux?
- Which filter has the most segmented sources?
- Which band has the highest contrast?
- Which image appears noisier or broader?

In [ ]:
comparison_df = metrics_df.copy()
comparison_df["flux_rank_desc"] = comparison_df["net_flux"].rank(ascending=False, method="dense")
comparison_df["std_rank_desc"] = comparison_df["std"].rank(ascending=False, method="dense")
comparison_df["source_rank_desc"] = comparison_df["n_segment_sources"].rank(ascending=False, method="dense")

display(comparison_df.sort_values("filter_name", key=lambda s: s.map(wavelength_key)))

In [ ]:
plt.figure(figsize=(8, 4))
ordered = comparison_df.sort_values("filter_name", key=lambda s: s.map(wavelength_key))
plt.bar(ordered["filter_name"], ordered["net_flux"])
plt.xlabel("Filter")
plt.ylabel("Approximate net flux")
plt.title("Approximate net flux by filter")
plt.grid(True, axis="y", alpha=0.3)
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(ordered["filter_name"], ordered["n_segment_sources"])
plt.xlabel("Filter")
plt.ylabel("Segmented source count")
plt.title("Detected segmented sources by filter")
plt.grid(True, axis="y", alpha=0.3)
plt.show()

## Step 9 — WCS alignment and multi-band image comparison

When WCS is present, we can reproject all chosen filters to one reference frame.

That lets us compare:
- morphology across wavelength
- pixel-by-pixel differences
- broad color / structure trends

In [ ]:
aligned = {}
selected_filters = sorted(list(loaded.keys()), key=wavelength_key)

if len(selected_filters) == 0:
    raise RuntimeError("No loaded filters available.")

reference_filter = selected_filters[0]
ref_header = loaded[reference_filter]["header"]
ref_shape = loaded[reference_filter]["data"].shape

for filt in selected_filters:
    item = loaded[filt]
    if filt == reference_filter:
        aligned[filt] = item["data"]
    else:
        try:
            reproj, footprint = reproject_interp((item["data"], item["wcs"]), ref_header, shape_out=ref_shape)
            aligned[filt] = robust_clean(reproj)
            print(f"Aligned {filt} -> {reference_filter}")
        except Exception as e:
            print(f"Could not align {filt}: {e}")

print("Aligned filters:", list(aligned.keys()))

In [ ]:
# Pairwise aligned correlation analysis
corr_rows = []
aligned_filters = sorted(aligned.keys(), key=wavelength_key)

for i, f1 in enumerate(aligned_filters):
    for f2 in aligned_filters[i+1:]:
        a = aligned[f1].ravel()
        b = aligned[f2].ravel()
        mask = np.isfinite(a) & np.isfinite(b)
        if mask.sum() > 10:
            corr = np.corrcoef(a[mask], b[mask])[0, 1]
        else:
            corr = np.nan
        corr_rows.append({"filter_1": f1, "filter_2": f2, "pixel_correlation": corr})

aligned_corr_df = pd.DataFrame(corr_rows)
display(aligned_corr_df)

In [ ]:
# Difference images relative to the reference filter
for filt in aligned_filters:
    if filt == reference_filter:
        continue
    diff = aligned[filt] - aligned[reference_filter]

    plt.figure(figsize=(7, 7))
    norm = simple_norm(diff, stretch="linear", percent=99.5)
    plt.imshow(diff, cmap="gray", norm=norm)
    plt.title(f"Difference image: {filt} - {reference_filter}")
    plt.axis("off")
    plt.show()

## Optional RGB quicklook

In [ ]:
if RGB_ENABLED and len(aligned) >= 1:
    rgb_assignment = auto_assign_rgb(list(aligned.keys()))
    print("RGB assignment:", rgb_assignment)

    r = normalize_channel(aligned[rgb_assignment["R"]])
    g = normalize_channel(aligned[rgb_assignment["G"]])
    b = normalize_channel(aligned[rgb_assignment["B"]])

    rgb = np.dstack([r, g, b])

    plt.figure(figsize=(10, 10))
    plt.imshow(rgb)
    plt.title("RGB quicklook")
    plt.axis("off")
    plt.show()

    if USE_LUPTON_RGB:
        try:
            lupton = make_lupton_rgb(
                aligned[rgb_assignment["R"]],
                aligned[rgb_assignment["G"]],
                aligned[rgb_assignment["B"]],
                stretch=5, Q=8
            )
            plt.figure(figsize=(10, 10))
            plt.imshow(lupton)
            plt.title("Lupton RGB quicklook")
            plt.axis("off")
            plt.show()
        except Exception as e:
            print("Lupton RGB failed:", e)

## Step 10 — Scientific interpretation helper

This cell builds a **plain-English report draft** from the measured analytics.

It is not a peer-reviewed interpretation.
It is a notebook-generated summary of what the data metrics suggest.

In [ ]:
def make_interpretation_text(metrics_df, target, reference_filter):
    lines = []
    lines.append(f"Target analyzed: {target}")
    lines.append(f"Reference filter for alignment: {reference_filter}")
    lines.append("")
    lines.append("High-level findings:")

    if len(metrics_df):
        by_flux = metrics_df.sort_values("net_flux", ascending=False)
        brightest_filter = by_flux.iloc[0]["filter_name"]
        faintest_filter = by_flux.iloc[-1]["filter_name"]

        by_sources = metrics_df.sort_values("n_segment_sources", ascending=False)
        most_structured = by_sources.iloc[0]["filter_name"]

        by_std = metrics_df.sort_values("std", ascending=False)
        noisiest_or_most_variable = by_std.iloc[0]["filter_name"]

        lines.append(f"- Brightest approximate aperture flux appears in filter: {brightest_filter}")
        lines.append(f"- Lowest approximate aperture flux appears in filter: {faintest_filter}")
        lines.append(f"- Largest segmentation source count appears in filter: {most_structured}")
        lines.append(f"- Largest global intensity spread (std) appears in filter: {noisiest_or_most_variable}")
        lines.append("")
        lines.append("How to read this:")
        lines.append("- Higher net flux can indicate the selected bright source is stronger in that band.")
        lines.append("- More segmented sources can indicate richer detectable structure or easier source separation.")
        lines.append("- Higher standard deviation can reflect stronger structure, contrast, or noise.")
        lines.append("- Cross-filter differences may reveal wavelength-dependent morphology.")
    else:
        lines.append("- No metrics were available.")

    return "\n".join(lines)

interpretation_text = make_interpretation_text(metrics_df, MAST_TARGET, reference_filter)
print(interpretation_text)

## Step 11 — Export report tables and summary files

In [ ]:
metrics_csv = OUTPUT_DIR / "jwst_filter_metrics.csv"
segment_csv = OUTPUT_DIR / "jwst_segment_sources.csv"
inventory_csv = OUTPUT_DIR / "jwst_catalog.csv"
manifest_csv = OUTPUT_DIR / "jwst_download_manifest.csv"
obs_csv = OUTPUT_DIR / "jwst_observations.csv"
product_csv = OUTPUT_DIR / "jwst_products_selected.csv"
corr_csv = OUTPUT_DIR / "jwst_aligned_correlations.csv"
summary_json = OUTPUT_DIR / "jwst_summary_report.json"
summary_txt = OUTPUT_DIR / "jwst_interpretation_report.txt"

metrics_df.to_csv(metrics_csv, index=False)
catalog.to_csv(inventory_csv, index=False)
download_manifest.to_csv(manifest_csv, index=False)
obs_df.to_csv(obs_csv, index=False)
download_products_df.to_csv(product_csv, index=False)

if len(segment_sources_df):
    segment_sources_df.to_csv(segment_csv, index=False)

if len(aligned_corr_df):
    aligned_corr_df.to_csv(corr_csv, index=False)

summary = {
    "target": MAST_TARGET,
    "radius_deg": MAST_RADIUS_DEG,
    "downloaded_files": [str(p) for p in ok_files],
    "filters": sorted(list(loaded.keys()), key=wavelength_key),
    "reference_filter": reference_filter,
    "n_observations_selected": int(len(obs_df)),
    "n_products_selected": int(len(download_products_df)),
    "n_downloaded_files": int(len(ok_files)),
    "metrics_columns": list(metrics_df.columns),
}

with open(summary_json, "w") as f:
    json.dump(summary, f, indent=2)

with open(summary_txt, "w") as f:
    f.write(interpretation_text)

print("Saved outputs:")
for p in [metrics_csv, inventory_csv, manifest_csv, obs_csv, product_csv, corr_csv, summary_json, summary_txt]:
    if p.exists():
        print("-", p)
if len(segment_sources_df) and segment_csv.exists():
    print("-", segment_csv)

## What this notebook helps you answer

After running the notebook, you should be able to answer questions like:

### Data inventory
- Which public JWST observations exist for this target?
- Which instruments and detectors were used?
- Which calibrated image products were available?

### Imaging analytics
- Which filter produced the strongest approximate source flux?
- Where is the brightest source in each image?
- How concentrated is the light profile?
- Are there detector row/column artifacts or gradients?
- Which band has the largest intensity spread?

### Source analytics
- How many segmented sources are detectable in each filter?
- Which filter reveals richer structure?
- Where are the brightest segmented sources located?

### Cross-band interpretation
- Which wavelengths emphasize different structures?
- How correlated are aligned bands?
- What changes appear in difference images?
- Which filters make the best RGB quicklook combination?

### Deliverables
- CSV tables for further analysis
- JSON summary report
- plain-English text interpretation
- reusable code for future targets

## Troubleshooting

### 1) MAST cannot be reached
That is usually a notebook-runtime network issue.

### 2) Search works but no useful products download
Try:
- smaller `MAST_MAX_OBS`
- smaller `PRODUCT_LIMIT`
- a different public target
- a wider radius if the target naming is sparse

### 3) FITS files download but no usable 2D science image is found
Some selected products may not be standard image science products. Try another target or tighten product filtering.

### 4) Segmentation finds too many or too few sources
Tune:
- `SEG_NSIGMA`
- `SEG_NPIXELS`

### 5) RGB looks washed out
Tune:
- `LOW_PERCENTILE`
- `HIGH_PERCENTILE`
- `STRETCH`

### 6) Alignment fails
Some products may have incomplete or incompatible WCS metadata.